In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import cv2
import numpy as np
import os
import shutil

BASE = '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images'
OUT = '/kaggle/working/dataset'

for split in ['train', 'val']:
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)

def convert_mask_to_yolo(mask_path, label_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    lines = []
    unique_classes = np.unique(mask)
    unique_classes = unique_classes[unique_classes != 0]
    for class_id in unique_classes:
        binary = (mask == class_id).astype(np.uint8)
        contours, _ = cv2.findContours(
            binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        for contour in contours:
            if cv2.contourArea(contour) < 100:
                continue
            points

In [ ]:
category_path = '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/category_id.txt'

names = []
with open(category_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            names.append(parts[1])

names = names[1:]  # remove background class

yaml_content = f"""path: /kaggle/working/dataset
train: images/train
val: images/val

nc: {len(names)}
names: {names}
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print(f"data.yaml created with {len(names)} classes")

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
print("Model loaded successfully")

In [ ]:
results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    device='0,1',
    patience=10,
    project='/kaggle/working/runs',
    name='platecalc'
)

In [ ]:
import glob
import shutil

pt_files = glob.glob('/kaggle/working/runs/**/best.pt', recursive=True)
if pt_files:
    shutil.copy(pt_files[0], '/kaggle/working/best.pt')
    print("Saved from:", pt_files[0])
else:
    print("No weights found!")